# Class 6: n8n — Build Your First AI Automation Workflow

**Tools:** n8n (n8n.io), Google Forms, Gmail, Telegram, Amplitude

### Agenda:
1. The **n8n canvas** — triggers, nodes, connections, and how data flows visually between them
2. The **5 core node types** every SWE should know — HTTP Request, Gmail, Google Sheets, Telegram, AI/LLM
3. Building your **first live workflow** together — Google Form submission → AI summary → Gmail
4. How to **read, trace, and debug** a workflow visually — the data bubble between every node
5. **Hands-on:** your own AI-powered automation (morning news digest → AI summary → Telegram)
6. Connecting **n8n or Zapier to Amplitude** — the product-analytics → AI → action pattern

> **Before class starts — prerequisites (5 minutes of setup):**
> - Sign up for **n8n Cloud** free tier at [n8n.io](https://n8n.io) *(or self-host if you prefer)*
> - You should already have a Google account (for Gmail + Sheets + Forms)
> - Get an **OpenAI, Anthropic, or Gemini API key** — for the AI node (free tiers work)

> - *Optional:* Create a **Telegram bot** via @BotFather (takes 2 minutes)

## Section 1 — Intro: The Tab-Swapping Problem (10 min)

> "Let me describe a scene and see if it feels familiar. Every morning, you open a dashboard. You copy a number or a piece of text. You paste it into ChatGPT. You ask it to summarise. You copy the summary. You open Gmail. You paste it. You send the email. Done. Every. Single. Day.
>
> That's not engineering. That's being a human clipboard. And in 2026, there is zero reason for a software engineer to be doing this.
>
> Today's class is about **n8n** — a visual automation tool that lets you chain together any API, any SaaS tool, and any LLM into a workflow that runs by itself. No backend to deploy. No cron jobs to maintain. No infrastructure. You build it in a canvas by clicking and dragging, and then it runs forever.
>
> By the end of today, you'll have built a workflow you actually use. Not a toy. Not a tutorial. Something that does a small, real, repetitive task for you automatically."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/555/original/1.png?1776665215)

**Why this matters for SWEs:**
- Most early-stage startups have **10+ n8n workflows running in production** — internal tooling, customer notifications, AI glue, ops dashboards
- Knowing n8n means you can ship an **AI-powered feature in 30 minutes** instead of 3 days of backend work
- It's the **fastest path to a working MVP** for anything that needs to connect 2+ services with an LLM in between
- Interview signal: *"I automate my own work"* reads very differently from *"I do the same thing manually every morning."*

**How this builds on previous classes:**
- Class 3's **prompt engineering** skills get reused — every AI node is a system prompt + input
- Class 4's **LLM platforms** (Gemini, Claude, OpenAI) — you'll pick one to power the AI node
- Class 5's **RAG** was about grounding AI in knowledge. Today is about putting AI into motion.

## Section 2 — The n8n Canvas: Three Ideas, That's It

> "Everything you will ever do in n8n reduces to three concepts. Learn these three and the rest of the tool is discovery, not learning."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/556/original/2.png?1776665636)

### One-line analogies:
- **Trigger** = the doorbell — it rings, and the whole workflow wakes up
- **Node** = a single room the data walks through — each room does one thing (call an API, summarise with AI, send an email)
- **Connection** = the corridor between rooms — data walks through it as JSON, getting richer as it goes

### The one technical thing you need to know

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/557/original/3.png?1776665782)

Data moves between nodes as **JSON** (JavaScript Object Notation — just a structured `{ "key": "value" }` format). Every node:
1. **Reads** the JSON from whichever node came before it
2. **Does its thing** (calls an API, runs an LLM prompt, writes a row)
3. **Outputs new JSON** that the next node can read

That's it. Every workflow you will ever build is a chain of this pattern.

**Beginner Note:** You don't have to write JSON by hand. n8n's UI lets you click on fields from the previous node and it fills in the reference for you.

**Intermediate Note:** You reference previous-node data using the `{{ $json.fieldName }}` expression syntax. Any field, any depth, any earlier node.

## Section 3 — Tool Demo: The 5 Core Node Types (20 min)

> "There are literally hundreds of nodes in n8n's catalogue — everything from Notion to Stripe to Shopify to Postgres. But 80% of all useful workflows use the same five. If you learn these five today, you can build almost anything you'll actually need."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/558/original/4.png?1776665879)

### Step-by-step walkthrough:

**Step 1 — Open n8n.**
Log into your n8n Cloud account. Click **"+ Create workflow"**. You'll see a blank canvas with a single **"Add first step"** button in the middle. Click it.

**Step 2 — Explore the node catalogue.**
A side panel opens with search + categories. Type each of the five node names below into the search bar to see the config panel for each.

**Node 1 — HTTP Request**
- The Swiss Army knife of nodes. Calls **any** REST API.
- Config: URL, method (GET/POST/etc.), headers, body, auth.
- *Example usage:* hit a news API, call OpenWeather, trigger a webhook somewhere else.

**Node 2 — Gmail**
- Requires connecting your Google account via OAuth (one-click — n8n walks you through).
- Operations: Send, Reply, Read, Draft.
- *Example usage:* send the output of an AI node as a well-formatted email.

**Node 3 — Google Sheets**
- Same Google OAuth as Gmail.
- Operations: Append row, Read rows, Update row, Delete.
- **Intermediate Note:** Treat Sheets as your free, shareable, UI-included database for any workflow where you don't need real scale. 80% of internal tools can be built this way.

**Node 4 — Telegram**
- Needs a Telegram bot token (create via @BotFather in Telegram — takes 2 minutes).
- Operations: Send message, Send photo, Send document.
- *Example usage:* AI summarises the news → Telegram sends it to you at 8am.

**Node 5 — AI Agent / OpenAI / Anthropic**
- Drop-in LLM call. Pick model, system prompt, user prompt, temperature.
- Input: whatever JSON arrived from the previous node. Output: the model's response.
- **Pro tip:** This is where your Class 3 prompt-engineering skills earn their keep. A vague prompt = unreliable workflow. A specific, structured prompt = workflow you can trust in production.

**Step 3 — Point out credentials.**
Top-right of every node config panel — the **Credentials** dropdown. You set up OAuth or API keys once, then reuse across all workflows. n8n encrypts and stores them.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/562/original/5.png?1776667601)

## Section 4 — Live Build: Form → AI Summary → Email (20 min)

> "Now we build. I'll drive, you follow. In 20 minutes we're going to ship a real workflow that takes a Google Form submission, asks AI to summarise it, and emails the summary to you automatically. This is the 'hello world' of AI automation — small enough to build in one sitting, meaningful enough that you'll actually keep using it."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/563/original/7.png?1776667733)

### Step-by-step build

**Step 1 — Create the form.**
Open [forms.google.com](https://forms.google.com). Make a simple form called *"Daily Standup"* with 3 fields:
- *Your name* (short answer)
- *What did you ship today?* (paragraph)
- *What's blocking you?* (paragraph)

Submit one test response so there's data in the system.

**Step 2 — Open n8n, create workflow.**
Name it: *"Daily Standup · AI Summary"*.

**Step 3 — Add the trigger: "On Form Submission".**
Search nodes for **Google Forms** → select the **"On form submission"** trigger. Connect your Google account. Select the form you just made. n8n will poll the form every minute by default.

**Step 4 — Execute the trigger once manually.**
Click **"Execute step"**. n8n pulls the most recent submission and shows you the JSON output. **Look at the JSON carefully** — memorise the field names. That's the shape of data flowing into the next node.

**Step 5 — Add the AI node.**
Click the `+` after the trigger. Search for **"Gemini"**. Select the **"Message a model"** operation.
- Connect your API key credential.
- Pick your model — `gemini-2.5-flash` all work fine and are cheap.
- **System prompt:** paste the template from the code cell above.
- **User prompt:** click the data-pill picker and drop in `{{ $json['What did you ship today?'] }}`.

**Step 6 — Test the AI node.**
Click **"Execute step"**. You should see the LLM's summary appear in the output panel as a string.

**Step 7 — Add the Gmail node.**
Click the `+` after the AI node. Search **Gmail** → **"Send a message"**. Connect your Gmail credentials.
- **To:** your own email address
- **Subject:** `Daily Standup Summary — {{ $json.name }}` *(referenced from the original form trigger via the data pill)*
- **Body:** `{{ $json.message.content }}` *(the AI's summary — field path depends on your chosen model)*

**Step 8 — Activate the workflow.**
Top-right of the canvas: flip the **Active** toggle. Now submit another form response. Wait ~1 minute. Check your inbox. **Email should be there, summarised, automatically.**

You just shipped an AI-powered feature. Zero backend. Zero code. Under 20 minutes.

## Section 5 — Guided Exercise: Reading, Tracing, Debugging (15 min)

> "Here's the best thing about n8n — when something breaks (and it will), you can *see* exactly where and why. No console logs. No stack traces. Just clicking on nodes and inspecting JSON."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/565/original/8.png?1776668167)

### The debugging loop in n8n — memorise this flow

1. **Execute the whole workflow** with test data.
2. If something went wrong, **click the red-bordered node** — the one that failed.
3. Look at its **Input panel** (left) — did it receive the JSON it expected?
4. If input is wrong → the problem is in a **previous** node. Walk backwards.
5. Look at its **Output panel** (right) — is the output shape what the next node needs?
6. **Fix the config**, re-execute that single node (not the whole workflow).
7. Repeat until green.

### Exercise 5.1 — Trace the Data

**In the workflow you just built in Section 4:**

1. Click on the **Form trigger** node. Look at the output JSON. Copy the exact field name for the "What did you ship today?" response.
2. Click on the **AI node**. Look at the input. **Does the field name match what you used in your prompt?** If not, that's a bug waiting to happen. Fix it with the data-pill picker.
3. Click on the **Gmail node**. Look at how the AI's response appears in its input.

**What you're training:** the habit of *reading JSON shapes at every node boundary*. This is 90% of n8n debugging.

### Exercise 5.2 — Break It On Purpose

**Try each of these and observe what n8n shows you:**

1. **Wrong field reference.** In the AI node, change the prompt to reference `{{ $json.nonexistent_field }}`. Execute. **What error does n8n show?** (Hint: it politely tells you the field was `undefined`.)
2. **Empty form submission.** Submit the form with all fields blank. Does your workflow handle it gracefully or crash? **This is a real production concern** — workflows need to handle edge cases.
3. **Remove the AI API credential.** Execute. See the error. Reattach. See the fix.

**Intermediate Note:** Right-click any node → **"Execute step"** runs only that one node using the most recent data. This is the single biggest speed-up for debugging. You don't need to re-run the whole chain every time.

**Advanced Note:** Use the **IF** node to add error branches. On the "error" path, send yourself a Telegram message. You've just built your first self-monitoring workflow.

## Section 6 — Mini Build: Your Own AI Automation (25 min)

> "Time to build something *you actually want*. Three challenges below. Pick one, ship it in 25 minutes. These are all small enough to finish and real enough to keep using after class."

### Challenge A: "The Morning Briefing" — News Digest to Telegram

**Scenario:** You want a personalised morning news digest every day at 8am. Not a generic newsletter — a 3-bullet summary of *exactly the topics you care about*, sent to your Telegram.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/567/original/10.png?1776669047)

**Your mission:** Build a scheduled workflow that runs every morning.

**Nodes to use:**
1. **Schedule Trigger** — set to fire daily at 8:00am local time.
2. **HTTP Request** — call a news API. Options:
   - Hacker News top stories: `https://hacker-news.firebaseio.com/v0/topstories.json`
   - NewsAPI (free tier): `https://newsapi.org/v2/top-headlines?country=us&apiKey=YOUR_KEY`
   - Your favourite RSS feed via the **RSS Read** node
3. **AI Node** — with this system prompt:

In [ ]:
# System prompt for "The Morning Briefing" AI node

# You are a tech-industry briefing writer for a busy software engineer.
#
# You will receive a JSON array of today's top stories (titles + URLs).
#
# ## Your task
# Pick the 3 stories most relevant to a working software engineer in 2026.
# Ignore celebrity news, politics, and generic clickbait.
#
# ## Output format — STRICT
# Exactly 3 lines. Each line formatted as:
#   • <2-sentence summary> — <URL>
#
# No preamble. No header. No sign-off. Just the three bullets.

4. **Telegram node** — send the AI output to your own chat ID.

**Success criteria:** Tomorrow at 8am, your phone buzzes with a clean 3-bullet summary. No generic newsletter. No banner ads. Just what you asked for.

**Pro tip:** Once it works, **add a second Schedule Trigger** at 6pm with a different prompt ("end-of-day wrap-up"). You now have a personal news agent running twice a day for free.

### Challenge B: "The Spreadsheet Sorter" — AI Categoriser

**Scenario:** You have a Google Sheet where new rows get added (feedback form, support tickets, job applications). Each row has unstructured text that a human would normally read, categorise, and flag. You want the AI to do that automatically the moment a row is added.

**Your mission:** Build a workflow that watches the sheet and writes AI-generated metadata back into it.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/568/original/11.png?1776669152)

**Nodes to use:**
1. **Google Sheets Trigger** — operation: *"On row added"*.
2. **AI Node** — system prompt: *"Read the text in the 'message' column. Output a JSON object with two keys: `category` (one of: bug, feature_request, praise, complaint, question) and `priority` (one of: low, medium, high, urgent). Output only valid JSON, no other text."*
3. **Google Sheets node** — operation: *"Update row"* — write the `category` and `priority` back into the same row.

**Success criteria:** Submit a new row manually. Within ~1 minute, the category and priority columns auto-fill. **You've built an internal tool that would take a human intern 2 hours a week — in 20 minutes, for free.**

**Intermediate Note:** This pattern (Sheet → AI → Sheet) is probably the single most common "ship it fast" AI automation in 2026 startups. Customer-feedback triage, lead scoring, content moderation — all variations of this exact shape.

### Challenge C: "The Amplitude Alert" — Product-Analytics AI Workflow

**Scenario:** Your product fires events to Amplitude ("user signed up", "user hit pricing page", "user dropped off at checkout"). Dashboards are great but nobody reads them. You want the **important events** to pierce through and reach the right person, with AI-generated context.

**Your mission:** Build a pattern where an Amplitude event triggers an AI-enriched alert.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/191/566/original/9.png?1776668961)

**Nodes to use:**
1. **Webhook Trigger** in n8n — copy the generated webhook URL.
2. In **Amplitude**, set up a **Destination** (Data → Destinations → Webhook) pointing to your n8n webhook. Filter to fire only on a specific event like *"Checkout Abandoned"*.
   - **No Amplitude account?** Don't worry — use n8n's **"Test webhook"** button and manually paste a sample Amplitude-style JSON payload. Everything downstream is identical.
3. **AI Node** — system prompt: *"You received a product analytics event. Write a 2-sentence Slack-ready message explaining (a) what the user did, and (b) what the most likely next step for the PM is. Be specific and actionable."*
4. **Slack node** (or **Email**, or **Telegram**) — send the AI's message to the right channel.

**Sample Amplitude payload to test with** (paste this into the webhook test panel):

In [ ]:
{
  "event_type": "Checkout Abandoned",
  "user_id": "usr_8374",
  "user_properties": {
    "plan": "pro_trial",
    "signup_days_ago": 12,
    "country": "IN"
  },
  "event_properties": {
    "cart_value_usd": 149,
    "step_abandoned": "payment_method",
    "time_on_page_sec": 94
  },
  "time": "2026-04-18T08:32:11Z"
}

**Success criteria:** Paste the sample payload → a Slack or Telegram message arrives with an AI-written summary like *"User usr_8374 on a Pro trial abandoned a $149 checkout at the payment step after 94s. Likely objection: payment method. Worth a one-off retention offer."*

**Why this challenge matters:** This exact pattern — **product event → AI context layer → human-actionable alert** — is the most in-demand automation in 2026 B2B SaaS. Every PM team wants it. Most engineering teams don't know how to ship it fast. You just did, in 20 minutes.

**Advanced Note — Zapier alternative:** The same pattern works in Zapier (Amplitude has a native Zapier integration). Trade-offs: Zapier is easier to set up but costs per task and has weaker AI nodes. For anything with branching logic or heavy AI use, n8n wins. For a one-shot prototype you want your non-technical PM to own, Zapier wins. Knowing when to reach for which is the real skill.

## Section 7 — Wrap-Up (10 min)

### The Decision Framework — When to Use What

decision framework: when to use ChatGPT vs Zapier vs n8n vs code

### What we built and learned today:

| Skill | What You Can Do Now |
|:---|:---|
| **n8n Canvas** | Read any workflow visually — trigger, nodes, connections, data flow |
| **Core 5 Nodes** | Build ~80% of useful automations using HTTP, Gmail, Sheets, Telegram, and AI nodes |
| **Live Build** | Ship a Form → AI → Email workflow end-to-end in under 20 minutes |
| **Visual Debugging** | Click any node to inspect input/output JSON; find and fix bugs by eye |
| **Your Own Workflow** | At least one AI-powered automation you built today that you'll keep using |
| **Amplitude Integration** | Connect product analytics events to AI-enriched, human-actionable alerts |

### Key takeaways

1. **n8n is the glue layer of the modern AI-powered stack.** Every time you find yourself copy-pasting between two apps with an LLM in the middle — that's an n8n workflow waiting to be built.

2. **Visual debugging is a superpower.** Stack traces are abstract. Clicking a node and seeing its JSON input and output is not. Use it.

3. **Start with n8n, graduate to code only when you must.** The right reason to leave n8n is *real* limits: scale, performance, custom UI. Not *"I want to write it in Python."*

4. **Prompts from Class 3 live inside n8n AI nodes.** Everything you learned about specific, structured, system-prompted LLM calls applies one-to-one. A vague prompt inside an automated workflow = an unreliable workflow.

5. **The Amplitude pattern generalises.** Stripe events, GitHub webhooks, Intercom triggers, Shopify orders — same shape. Event source → n8n webhook → AI → human-actionable action. Memorise the shape, instantiate it for whatever stack you're on.

6. **You can ship AI features now.** In 2026, a SWE who can build, debug, and ship automation workflows is strictly more valuable than one who can't. Today's class was not theory — it was one of the highest-leverage skills in your toolkit.

*Remember: The goal isn't to learn every node in the catalogue. It's to see a repetitive task and know — with zero hesitation — that you can automate it away in an afternoon.*